# Phase 2 — Network Construction
**Steps 2.1 – 2.5** | Build 11 NetworkX weighted graphs (5 temporal + 6 issue-specific).

**Methodology:** Temporal networks use pre-computed agreement S-scores from Harvard Dataverse
(Voeten et al.) — the standard in the IR literature. Issue-specific networks use Pearson
correlation with per-issue θ sweep (knee detection) since Dataverse S-scores are not
available at issue level.

Key decision: **θ = 0.70** for temporal networks (S-score threshold).

In [ ]:
import os, pickle
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
from itertools import combinations

matplotlib_style = {'figure.facecolor': '#0d1117', 'axes.facecolor': '#161b22',
                    'axes.edgecolor': '#30363d', 'text.color': 'white',
                    'axes.labelcolor': 'white', 'xtick.color': 'white',
                    'ytick.color': 'white', 'figure.dpi': 150}
plt.rcParams.update(matplotlib_style)

ROOT   = os.path.abspath(os.path.join(os.getcwd(), '..'))
PROC   = os.path.join(ROOT, 'data', 'processed')
EXT    = os.path.join(ROOT, 'data', 'external')
NETS   = os.path.join(ROOT, 'results', 'networks')
PLOTS  = os.path.join(ROOT, 'results', 'plots')
TABLES = os.path.join(ROOT, 'results', 'tables')
os.makedirs(NETS, exist_ok=True)
print('Root:', ROOT)

## Step 2.1 – Build Pairwise Agreement Matrices

In [ ]:
# ── Load Dataverse S-scores for temporal networks ──────────────
DV = os.path.join(ROOT, 'dataverse_files')
ERAS = ['full', 'cold_war', 'post_cw', 'post_9_11', 'recent']
ERA_YEARS = {
    'full': (1946, 2015), 'cold_war': (1946, 1991),
    'post_cw': (1991, 2001), 'post_9_11': (2001, 2014), 'recent': (2014, 2015)
}

# Load S-scores + ccode mapping
agree_df = pd.read_csv(os.path.join(DV, 'AgreementScores.csv'),
                        usecols=['ccode1','ccode2','agree','year'])
votes_clean = pd.read_csv(os.path.join(PROC, 'votes_clean.csv'), usecols=['ccode','country'])
ccode_map = dict(votes_clean.drop_duplicates('ccode')[['ccode','country']].values)
agree_df['country_a'] = agree_df['ccode1'].map(ccode_map)
agree_df['country_b'] = agree_df['ccode2'].map(ccode_map)
agree_df = agree_df.dropna(subset=['country_a','country_b','agree'])
print(f'Loaded {len(agree_df):,} dyad-year S-scores, {agree_df["country_a"].nunique()} countries')
print(f'S-score: mean={agree_df["agree"].mean():.3f}, median={agree_df["agree"].median():.3f}')

def build_agreement_matrix_sscore(agree_sub):
    """Symmetric agreement matrix from Dataverse S-scores (averaged over era)."""
    dyad_mean = agree_sub.groupby(['country_a','country_b'])['agree'].mean().reset_index()
    countries = sorted(set(dyad_mean['country_a']) | set(dyad_mean['country_b']))
    n = len(countries)
    idx = {c: i for i, c in enumerate(countries)}
    mat = np.full((n, n), np.nan)
    np.fill_diagonal(mat, 1.0)
    for _, row in dyad_mean.iterrows():
        i, j = idx.get(row['country_a']), idx.get(row['country_b'])
        if i is not None and j is not None:
            mat[i,j] = row['agree']; mat[j,i] = row['agree']
    return pd.DataFrame(mat, index=countries, columns=countries)

def build_agreement_matrix_pearson(df, min_votes=5):
    """Pearson correlation — used only for issue networks."""
    pivot = df.pivot_table(index='rcid', columns='country', values='v', aggfunc='first')
    pivot = pivot.dropna(axis=1, thresh=min_votes)
    return pivot.corr(method='pearson', min_periods=min_votes)

era_matrices = {}
for era in ERAS:
    y0, y1 = ERA_YEARS[era]
    sub = agree_df[(agree_df['year'] >= y0) & (agree_df['year'] <= y1)]
    era_matrices[era] = build_agreement_matrix_sscore(sub)
    print(f'{era}: matrix shape = {era_matrices[era].shape}')
era_matrices['full'].to_csv(os.path.join(PROC, 'agree_matrix_full.csv'))

## Step 2.2 – Threshold Selection (Critical Decision)

In [ ]:
def threshold_sweep(agree_matrix, thresholds=np.arange(0.3, 0.96, 0.05)):
    """Sweep agreement thresholds and record graph stats."""
    countries = agree_matrix.columns.tolist()
    n = len(countries)
    results = []
    for theta in thresholds:
        G = nx.Graph()
        G.add_nodes_from(countries)
        for i in range(n):
            for j in range(i+1, n):
                w = agree_matrix.iloc[i, j]
                if not np.isnan(w) and w >= theta:
                    G.add_edge(countries[i], countries[j], weight=float(w))
        if G.number_of_edges() == 0:
            results.append({'theta': theta, 'n_nodes': n, 'n_edges': 0,
                            'giant_frac': 0, 'density': 0})
            continue
        gcc = max(nx.connected_components(G), key=len)
        giant_frac = len(gcc) / n
        results.append({
            'theta': theta,
            'n_nodes': n,
            'n_edges': G.number_of_edges(),
            'giant_frac': giant_frac,
            'density': nx.density(G)
        })
    return pd.DataFrame(results)

print('Running threshold sweep on full network...')
sweep_df = threshold_sweep(era_matrices['full'])

# Find percolation knee: where giant_frac first drops below 0.8
knee_row = sweep_df[sweep_df['giant_frac'] < 0.80].head(1)
KNEE_THETA = knee_row['theta'].values[0] if not knee_row.empty else 0.70
print(f'\nPercolation knee detected at μ = {KNEE_THETA}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
fig.suptitle('Threshold Sweep — Full Network (1946–2015)', color='white', fontsize=13)

axes[0].plot(sweep_df['theta'], sweep_df['giant_frac'], 'o-', color='#58a6ff', lw=2)
axes[0].axvline(KNEE_THETA, color='#f85149', ls='--', label=f'θ = {KNEE_THETA} (knee)')
axes[0].set_xlabel('Threshold θ'); axes[0].set_ylabel('Giant component fraction')
axes[0].set_title('Giant Component Size vs θ', color='white'); axes[0].legend()

axes[1].plot(sweep_df['theta'], sweep_df['n_edges'], 's-', color='#3fb950', lw=2)
axes[1].axvline(KNEE_THETA, color='#f85149', ls='--', label=f'θ = {KNEE_THETA}')
axes[1].set_xlabel('Threshold θ'); axes[1].set_ylabel('Number of edges')
axes[1].set_title('Edge Count vs θ', color='white'); axes[1].legend()

plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p2_threshold_sweep.png'), bbox_inches='tight', facecolor='#0d1117')
plt.show()

print('\nThreshold sweep results:')
print(sweep_df[sweep_df['theta'].isin([0.50, 0.60, 0.70, 0.75, 0.80, 0.85, 0.90])].to_string(index=False))

In [ ]:
THETA = 0.70   # Hard-coded based on percolation sweep (KNEE_THETA)
print(f'Chosen global threshold: θ = {THETA}')

## Step 2.3 – Build Full + Temporal Graphs

In [ ]:
# Load external attributes
region_df = pd.read_csv(os.path.join(EXT, 'un_regional_groups.csv'))
income_df = pd.read_csv(os.path.join(EXT, 'wb_income_groups.csv'))
region_map = dict(zip(region_df['country'], region_df['region']))
income_map = dict(zip(income_df['country'], income_df['income_group']))

def build_graph(corr_matrix, theta, label=''):
    """Build a weighted NetworkX graph from a correlation matrix."""
    countries = corr_matrix.index.tolist()
    G = nx.Graph(name=label)
    for c in countries:
        G.add_node(c,
                   region=region_map.get(c, 'Unknown'),
                   income=income_map.get(c, 'Unknown'))
    n = len(countries)
    for i in range(n):
        for j in range(i+1, n):
            w = corr_matrix.iloc[i, j]
            if not np.isnan(w) and w >= theta:
                G.add_edge(countries[i], countries[j], weight=float(w))
    return G

era_graphs = {}
stats_rows = []

for era in ERAS:
    G = build_graph(era_matrices[era], THETA, label=era)
    era_graphs[era] = G
    n = G.number_of_nodes()
    m = G.number_of_edges()
    gcc = max(nx.connected_components(G), key=len)
    stats_rows.append({
        'network': era, 'nodes': n, 'edges': m,
        'density': round(nx.density(G), 4),
        'avg_degree': round(2*m/n, 2) if n > 0 else 0,
        'giant_component': len(gcc),
        'components': nx.number_connected_components(G)
    })
    print(f'{era:15s}: {n} nodes, {m} edges, density={nx.density(G):.4f}')

    # Sanity check
    pivot_countries = era_matrices[era].columns
    for key_country in ['United States of America', 'China', 'Russia', 'India']:
        if key_country in G:
            deg = G.degree(key_country)
            print(f'  {key_country}: degree = {deg}')

## Step 2.4 – Build 6 Issue-Specific Graphs

In [ ]:
ISSUE_MAP = {
    'me': 'israel_palestine', 'co': 'colonization',
    'hr': 'human_rights',     'di': 'disarmament',
    'nu': 'nuclear_weapons',  'ec': 'economic_development'
}

full_df = pd.read_csv(os.path.join(PROC, 'votes_full.csv'))
print('Full df columns:', full_df.columns.tolist()[:15])

def find_knee_theta(mat, thetas=np.arange(0.30, 0.96, 0.05)):
    """Per-issue theta sweep — find the knee."""
    countries = mat.columns.tolist(); n = len(countries)
    fracs = []
    for theta in thetas:
        G = nx.Graph(); G.add_nodes_from(countries)
        for i in range(n):
            for j in range(i+1, n):
                w = mat.iloc[i,j]
                if not np.isnan(w) and w >= theta: G.add_edge(countries[i], countries[j])
        fracs.append(len(max(nx.connected_components(G), key=len))/n if G.number_of_edges() > 0 else 0)
    fracs = np.array(fracs)
    d2 = np.diff(fracs, n=2)
    return round(thetas[np.argmin(d2)+1], 2)

issue_graphs = {}
for short, col in ISSUE_MAP.items():
    if short not in full_df.columns:
        print(f'  WARNING: column {short} not found'); continue
    sub = full_df[full_df[short] == 1].copy()
    print(f'\nIssue {short}: {sub["rcid"].nunique()} resolutions, {sub["country"].nunique()} countries')
    mat = build_agreement_matrix_pearson(sub)
    theta_iss = max(find_knee_theta(mat), 0.30)
    G_iss = build_graph(mat, theta_iss, label=f'issue_{short}')
    issue_graphs[short] = G_iss
    print(f'  theta={theta_iss}, N={G_iss.number_of_nodes()}, M={G_iss.number_of_edges()}')
    stats_rows.append({
        'network': f'issue_{short}', 'nodes': G_iss.number_of_nodes(),
        'edges': G_iss.number_of_edges(), 'density': round(nx.density(G_iss), 4),
        'avg_degree': round(2*G_iss.number_of_edges()/max(G_iss.number_of_nodes(),1), 2),
        'giant_component': len(max(nx.connected_components(G_iss), key=len)) if G_iss.number_of_edges() > 0 else 0,
        'components': nx.number_connected_components(G_iss),
        'method': 'Pearson', 'theta': theta_iss
    })

## Step 2.5 – Export & Validate

In [ ]:
all_graphs = {**era_graphs, **{f'issue_{k}': v for k, v in issue_graphs.items()}}

for name, G in all_graphs.items():
    # GraphML export
    gml_path = os.path.join(NETS, f'{name}.graphml')
    nx.write_graphml(G, gml_path)
    # Pickle export
    pkl_path = os.path.join(NETS, f'{name}.pkl')
    with open(pkl_path, 'wb') as f:
        pickle.dump(G, f)

print(f'Exported {len(all_graphs)} graphs to {NETS}')

# Summary stats table
stats_df = pd.DataFrame(stats_rows)
stats_path = os.path.join(TABLES, 'p2_network_stats.csv')
stats_df.to_csv(stats_path, index=False)
print('\nNetwork statistics:')
print(stats_df.to_string(index=False))

In [ ]:
# Visualise full network (spring layout, colored by region)
G_full = era_graphs['full']

REGION_COLORS = {
    'Africa': '#f85149', 'AsiaPacific': '#58a6ff', 'EasternEurope': '#3fb950',
    'GRULAC': '#d2a8ff', 'WEOG': '#ffa657', 'ArabStates': '#79c0ff', 'Unknown': '#8b949e'
}

node_colors = [REGION_COLORS.get(G_full.nodes[n].get('region', 'Unknown'), '#8b949e')
               for n in G_full.nodes]
edge_weights = [G_full[u][v]['weight'] for u, v in G_full.edges]

print('Computing layout (this may take ~30s for large networks)...')
pos = nx.spring_layout(G_full, seed=42, k=0.3, iterations=50)

fig, ax = plt.subplots(figsize=(16, 12))
fig.patch.set_facecolor('#0d1117')
ax.set_facecolor('#0d1117')

nx.draw_networkx_edges(G_full, pos, ax=ax,
                       edge_color=[w for w in edge_weights],
                       edge_cmap=plt.cm.Blues,
                       alpha=0.3, width=0.5)
nx.draw_networkx_nodes(G_full, pos, ax=ax,
                       node_color=node_colors, node_size=30, alpha=0.9)

# Label only major powers
major_powers = {n: n.split()[-1] for n in ['United States of America','China','Russia',
                                             'India','Brazil','Germany','France','Japan',
                                             'South Africa','Nigeria'] if n in G_full}
nx.draw_networkx_labels(G_full, pos, labels=major_powers, ax=ax,
                        font_size=7, font_color='white')

# Legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor=c, label=r) for r, c in REGION_COLORS.items() if r != 'Unknown']
ax.legend(handles=legend_elements, loc='lower left', fontsize=8,
          facecolor='#161b22', edgecolor='#30363d', labelcolor='white')
ax.set_title('UNGA Voting Network — Full Era (1946–2015)\nColored by UN Regional Group',
             color='white', fontsize=14)
ax.axis('off')
plt.tight_layout()
plt.savefig(os.path.join(PLOTS, 'p2_full_network.png'), bbox_inches='tight',
            facecolor='#0d1117', dpi=180)
plt.show()
print('Saved full network plot')

## ✅ Phase 2 Complete
**Outputs:**
- 11 NetworkX graphs (5 temporal + 6 issue-specific) as `.graphml` + `.pkl`
- `results/tables/p2_network_stats.csv`
- `results/plots/p2_threshold_sweep.png`
- `results/plots/p2_full_network.png`

**→ Proceed to Notebook 03: Topology Analysis**